In [2]:
import sys
import featuregraph as fg

print("Python:", sys.executable)
print("FeatureGraph:", fg.__file__)

Python: /usr/local/bin/python
FeatureGraph: /workspaces/featuregraph/src/featuregraph/__init__.py


In [3]:
import sys
import featuregraph as fg

print("Python:", sys.executable)
print("FeatureGraph:", fg.__file__)

Python: /usr/local/bin/python
FeatureGraph: /workspaces/featuregraph/src/featuregraph/__init__.py


In [4]:
assert callable(fg.oscillate)

print("fg.oscillate is available")

fg.oscillate is available


In [5]:
import sys

loaded_state_detection_modules = [
    name
    for name in sys.modules
    if name == "state_detection"
    or name.startswith("state_detection.")
]

assert not loaded_state_detection_modules, (
    "state_detection modules were imported: "
    f"{loaded_state_detection_modules}"
)

print("No state_detection modules are loaded")

No state_detection modules are loaded


In [6]:
from pathlib import Path

featuregraph_root = Path(fg.__file__).resolve().parent

loaded_featuregraph_modules = {
    name: module.__file__
    for name, module in sys.modules.items()
    if name == "featuregraph"
    or name.startswith("featuregraph.")
    if getattr(module, "__file__", None)
}

for name, path in sorted(loaded_featuregraph_modules.items()):
    resolved = Path(path).resolve()

    assert featuregraph_root in resolved.parents or resolved == featuregraph_root, (
        f"{name} was loaded from an unexpected location: {resolved}"
    )

print("All loaded FeatureGraph modules come from:")
print(featuregraph_root)

All loaded FeatureGraph modules come from:
/workspaces/featuregraph/src/featuregraph


In [7]:
from pathlib import Path

source_root = Path("/workspaces/featuregraph/src/featuregraph")

matches = []

for path in source_root.rglob("*.py"):
    text = path.read_text(encoding="utf-8")

    if "state_detection" in text:
        matches.append(path)

assert not matches, (
    "Old state_detection references remain in:\n"
    + "\n".join(str(path) for path in matches)
)

print("No state_detection references found in FeatureGraph source")

No state_detection references found in FeatureGraph source


In [8]:
from featuregraph.utils.bidmc import load_bidmc_subject
from featuregraph.utils.eastman import load_tep_run
from featuregraph.data_formatting.rename_map import (
    bidmc_map,
    eastman_map,
)

eastman = (
    load_tep_run(
        "faulty_training",
        fault_number=1,
        simulation_run=1,
    )
    .rename(columns=eastman_map)
)

bidmc = (
    load_bidmc_subject(1)
    .rename(columns=bidmc_map)
)

print("Eastman:", eastman.shape)
print("BIDMC:", bidmc.shape)

Eastman: (3001, 57)
BIDMC: (60001, 7)


In [9]:
required_eastman = {
    "fault_number",
    "simulation_run",
    "reactor_pressure",
}

required_bidmc = {
    "subject",
    "respiration",
}

missing_eastman = required_eastman.difference(eastman.columns)
missing_bidmc = required_bidmc.difference(bidmc.columns)

assert not missing_eastman, (
    f"Missing Eastman columns: {sorted(missing_eastman)}"
)

assert not missing_bidmc, (
    f"Missing BIDMC columns: {sorted(missing_bidmc)}"
)

print("Both datasets have the required FeatureGraph columns")

Both datasets have the required FeatureGraph columns


In [10]:
eastman_oscillations = fg.oscillate(
    eastman,
    signal="reactor_pressure",
    group=["fault_number", "simulation_run"],
    smooth=True,
)

bidmc_oscillations = fg.oscillate(
    bidmc,
    signal="respiration",
    group=["subject"],
    smooth=False,
)

print("Eastman oscillations:", eastman_oscillations.shape)
print("BIDMC oscillations:", bidmc_oscillations.shape)

Eastman oscillations: (58, 10)
BIDMC oscillations: (555, 10)


In [11]:
expected_schema = [
    "oscillation_id",
    "start_index",
    "peak_index",
    "end_index",
    "rise_duration",
    "fall_duration",
    "duration",
    "period",
    "amplitude",
    "temporal_symmetry",
]

assert eastman_oscillations.columns.tolist() == expected_schema
assert bidmc_oscillations.columns.tolist() == expected_schema

print("Both datasets produce the expected object schema")

Both datasets produce the expected object schema


In [12]:
assert (
    eastman_oscillations.columns.tolist()
    == bidmc_oscillations.columns.tolist()
)

print("Cross-domain schema is identical")

Cross-domain schema is identical


In [13]:
def validate_oscillation_table(table, name):
    required = {
        "oscillation_id",
        "start_index",
        "peak_index",
        "end_index",
        "rise_duration",
        "fall_duration",
        "duration",
        "period",
        "amplitude",
        "temporal_symmetry",
    }

    assert required.issubset(table.columns)
    assert not table.empty
    assert table["oscillation_id"].is_unique

    complete = table.dropna(
        subset=["start_index", "peak_index", "end_index"]
    )

    assert (complete["start_index"] <= complete["peak_index"]).all()
    assert (complete["peak_index"] <= complete["end_index"]).all()
    assert (complete["duration"] >= 0).all()
    assert (complete["amplitude"] >= 0).all()

    bounded_symmetry = complete["temporal_symmetry"].dropna()

    assert bounded_symmetry.between(0, 1).all()

    print(
        f"{name}: {len(table)} objects, "
        f"{len(complete)} complete"
    )


validate_oscillation_table(
    eastman_oscillations,
    "Eastman",
)

validate_oscillation_table(
    bidmc_oscillations,
    "BIDMC",
)

Eastman: 58 objects, 57 complete
BIDMC: 555 objects, 554 complete


In [14]:
loaded_state_detection_modules = [
    name
    for name in sys.modules
    if name == "state_detection"
    or name.startswith("state_detection.")
]

assert not loaded_state_detection_modules, (
    "state_detection was loaded during the workflow: "
    f"{loaded_state_detection_modules}"
)

print("FeatureGraph workflow completed without state_detection")

FeatureGraph workflow completed without state_detection


In [15]:
import subprocess
import textwrap

code = textwrap.dedent(
    """
    import sys
    import featuregraph as fg

    from featuregraph.utils.bidmc import load_bidmc_subject
    from featuregraph.utils.eastman import load_tep_run
    from featuregraph.data_formatting.rename_map import (
        bidmc_map,
        eastman_map,
    )

    eastman = (
        load_tep_run(
            "faulty_training",
            fault_number=1,
            simulation_run=1,
        )
        .rename(columns=eastman_map)
    )

    bidmc = (
        load_bidmc_subject(1)
        .rename(columns=bidmc_map)
    )

    eastman_table = fg.oscillate(
        eastman,
        signal="reactor_pressure",
        group=["fault_number", "simulation_run"],
        smooth=True,
    )

    bidmc_table = fg.oscillate(
        bidmc,
        signal="respiration",
        group=["subject"],
        smooth=False,
    )

    assert eastman_table.columns.tolist() == bidmc_table.columns.tolist()

    assert not any(
        name == "state_detection"
        or name.startswith("state_detection.")
        for name in sys.modules
    )

    print("Clean-process self-containment test passed")
    """
)

result = subprocess.run(
    [sys.executable, "-c", code],
    capture_output=True,
    text=True,
    timeout=300,
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0

Clean-process self-containment test passed


